### 処理の流れ

このNotebookでは、ユーザーの質問をそのままローカルLLMへ渡すのではなく、必要に応じてWeb検索や回答チェックを組み合わせる。

```text
ユーザー入力
    ↓
日付表現を具体的な日付へ変換
    ↓
Web検索が必要か判定
    ↓
必要な場合だけTavilyで検索
    ↓
Qwen3-1.7Bで初回回答を生成
    ↓
Qwen3-4BでReflection
    ↓
問題があればQwen3-4Bで再生成
    ↓
最終回答を表示して会話履歴へ保存
```

軽量モデルを通常処理に使い、より大きいモデルを品質確認と再生成に使うことで、処理速度と回答品質の両立を試している。


In [1]:
from datetime import datetime, timedelta
import time

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Tavily API Keyは環境変数として事前に設定しています。API Key自体はNotebookには記載していません。
from tavily import TavilyClient


C:\ProgramData\anaconda3\envs\qwen3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 現在日付の取得

「今日」「昨日」「一昨日」「明日」などの相対的な日付表現を正しく扱えるように、基準日時を`reference_datetime`として取得する。

`current_date`や相対日付の計算は、すべて同じ`reference_datetime`を基準にする。

チャットループ内でも質問ごとに基準日時を更新するため、Notebookを日付をまたいで起動したままでも現在日付を再取得できる。


In [2]:
reference_datetime = datetime.now()
current_date = reference_datetime.strftime("%Y年%m月%d日")

print(current_date)


2026年09月01日


#### モデルの呼び出しと量子化

処理内容によってモデルを使い分ける。

- 軽量モデル `Qwen3-1.7B`：Web検索要否判定と初回回答生成
- 評価モデル `Qwen3-4B`：Reflectionと、FAIL時の再生成

通常処理は軽量モデルで速度を確保し、品質確認と修正だけ比較的大きなモデルへ任せる。

両モデルとも4bit量子化を使用する。

- `load_in_4bit=True`：4bit量子化
- `nf4`：4bit量子化方式
- `bfloat16`：量子化した値を使った計算時のデータ型
- `use_double_quant=True`：量子化に必要な情報もさらに量子化してメモリ使用量を削減


In [3]:
light_model_name = "Qwen/Qwen3-1.7B"
review_model_name = "Qwen/Qwen3-4B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)


In [4]:
# 軽量モデル：検索判定・初回回答生成
light_tokenizer = AutoTokenizer.from_pretrained(light_model_name)

light_model = AutoModelForCausalLM.from_pretrained(
    light_model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

# 評価モデル：Reflection・再生成
review_tokenizer = AutoTokenizer.from_pretrained(review_model_name)

review_model = AutoModelForCausalLM.from_pretrained(
    review_model_name,
    quantization_config=quantization_config,
    device_map="auto"
)


W0901 21:23:06.517000 6576 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 398/398 [00:06<00:00, 60.55it/s]


#### ここまでの流れ

```text
ライブラリのインポート
        ↓
軽量モデル Qwen3-1.7B を指定
        ↓
評価モデル Qwen3-4B を指定
        ↓
4bit量子化設定
        ↓
各Tokenizer / Modelを読み込み
```

検索判定と通常回答は1.7B、ReflectionとFAIL時の再生成は4Bを使用する。


### Qwen3単体での動作確認

Web検索機能を追加する前に、まずローカルLLMであるQwen3単体で文章生成が正常に行えるか確認する。

今回の処理の流れは以下の通り。

1. `messages` にユーザーの質問を定義する
2. `apply_chat_template()` でQwen3が扱えるチャット形式へ変換する
3. Tokenizerで文字列をToken IDへ変換する
4. Token IDをモデルへ渡して回答を生成する
5. 入力部分を除外し、Qwen3が新しく生成したTokenだけを取得する
6. Token IDを文字列へ戻して回答を表示する

#### 1. ユーザーの質問を定義

```python
messages = [
    {
        "role": "user",
        "content": "日本の首都はどこですか？"
    }
]
```

`messages` は会話内容を保持するリストである。

`role` には発言者、`content` には実際の発言内容を格納する。

今回はQwen3単体の動作確認が目的なので、ユーザーからの質問を1件だけ格納している。

---

#### 2. Chat Templateへ変換

```python
text = light_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
```

`apply_chat_template()` は、`messages` の内容をQwen3が会話として解釈できる形式へ変換する処理である。

`tokenize=False` とすることで、この段階ではToken IDへ変換せず、文字列として取得する。

`add_generation_prompt=True` は、会話の末尾に「ここからAssistantが回答する」という情報を追加するための設定である。

---

#### 3. Token IDへ変換

```python
inputs = light_tokenizer(
    text,
    return_tensors="pt"
).to(light_model.device)
```

Tokenizerを使用して、文字列をモデルが扱えるToken IDへ変換する。

`return_tensors="pt"` を指定することで、PyTorchのTensor形式で取得する。

`.to(model.device)` によって、入力データをモデルと同じデバイスへ配置する。

---

#### 4. Qwen3で回答を生成

```python
outputs = light_model.generate(
    **inputs,
    max_new_tokens=128
)
```

`model.generate()` にToken IDを渡し、Qwen3に回答を生成させる。

`max_new_tokens=128` は、モデルが新しく生成できるToken数の上限を128に設定している。

`outputs` には、入力TokenとQwen3が新しく生成したTokenが連結された状態で格納される。

イメージとしては以下のようになる。

```text
[入力Token][入力Token] ... [生成Token][生成Token] ...
```

---

#### 5. 生成された回答部分だけを取得

```python
generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
```

`outputs[0]` は、1件目の生成結果を取得している。

一方、

```python
inputs["input_ids"].shape[1]
```

では、入力として使用したToken数を取得している。

例えば入力が14Tokenの場合、

```text
outputs[0]

[入力 14Token][Qwen3が生成したToken]
              ↑
          ここから取得
```

となる。

そのため、

```python
outputs[0][14:]
```

のようにスライスすることで、先頭の入力Tokenを除外し、Qwen3が新しく生成した回答部分だけを取得できる。

---

#### 6. Token IDを文字列へ戻す

```python
response = light_tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
)
```

モデルが生成したToken IDを、人間が読める文字列へ変換する。

`skip_special_tokens=True` を指定することで、モデル内部で使用される特殊Tokenを表示結果から除外する。

最後に、

```python
print(response)
```

として生成結果を確認する。

```
<think>
Okay, the user is asking where Tokyo is, which is the capital of Japan. I need to confirm that Tokyo is indeed the capital. I remember that Tokyo is the capital, but I should double-check. Let me think... Yes, Japan's capital is Tokyo, and it's the largest city. I should mention that it's the political, economic, and cultural center. Also, maybe add some context about its population and significance. I should make sure the answer is accurate and concise. Let me structure it: first state that Tokyo is the capital, then mention its status as the political and economic center, and maybe touch
```

この段階ではWeb検索は使用しておらず、ローカル環境上のQwen3だけで回答を生成している。



**補足：** 今回はQwen3単体で文章生成が正常に動作することを確認するためのテストであり、出力言語や回答内容の調整は行っていない。そのため、生成結果に英語およびThinking部分が含まれているが、この段階ではそのままとしている。また、`max_new_tokens=128`として生成Token数を制限しているため、生成結果は途中で終了している。


In [5]:
messages = [
    {
        "role":"user",
        "content":"日本の首都はどこですか？"
    }
]
        

text = light_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = light_tokenizer(
    text,
    return_tensors="pt",
).to(light_model.device)

outputs = light_model.generate(
    **inputs,
    max_new_tokens=128,

)

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]

response = light_tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
)

print(response)


<think>
Okay, the user is asking where the capital of Japan is. I know that Japan's capital is Tokyo. But I should make sure I'm correct. Let me think. Japan is a country in East Asia, and the capital is indeed Tokyo. It's the largest city and the political, economic, and cultural center. I should confirm the details. The Emperor of Japan is in Tokyo, and the government is based there. Also, Tokyo is the most populous city in the world. I should mention that Tokyo is the capital and that it's the largest city. I should keep the answer straightforward and accurate.
</think>

日本の


### TavilyによるWeb検索の動作確認

Qwen3とWeb検索を組み合わせる前に、Tavily単体でWeb検索を実行し、検索結果がどのようなデータ構造で取得されるか確認する。

```python
tavily = TavilyClient()

search_result = tavily.search(
    query="今日の東京の天気",
    max_results=3
)
```

`TavilyClient()` でTavilyのクライアントを作成し、`search()` メソッドを使用してWeb検索を実行する。

`query` には検索する文字列を指定する。

`max_results=3` とすることで、取得する検索結果の最大件数を3件に設定している。

検索結果は辞書形式で返され、`results` の中に複数の検索結果がリストとして格納されている。

イメージとしては以下のような構造になる。

```text
search_result
│
├─ query
│
└─ results
     │
     ├─ [0]
     │   ├─ title
     │   ├─ url
     │   └─ content
     │
     ├─ [1]
     │   ├─ title
     │   ├─ url
     │   └─ content
     │
     └─ [2]
         ├─ title
         ├─ url
         └─ content
```

今回はデータ構造を確認するため、1件目の検索結果から`title`、`url`、`content`を取得する。

```python
print(search_result['results'][0]['title'])
print(search_result['results'][0]['url'])
print(search_result['results'][0]['content'])
```

`search_result['results']`で検索結果のリストを取得し、`[0]`でその1件目を取得している。

さらに、

* `['title']`：検索結果のタイトル
* `['url']`：検索元のURL
* `['content']`：検索結果から取得された本文情報

をそれぞれ取得する。

```
title:
東京都 東京の天気 - 毎日新聞


url:
https://mainichi.jp/weather/forecast/city/city_03_01_01.html


content:
# 毎日新聞

 全国の天気
 週間天気予報
 アメダス実況
 降水短時間予報
 衛星雲画像
 天気図
 洗濯情報
 台風
 地震
 警報・注意報
 世界の天気
 花粉情報

# 東京都 東京の天気

8月30日17時発表

## 今日の天気 - 8月30日(日)

|  |  |  |
 --- 
| 曇 曇 | 最高気温  - | 最低気温  - |

|  |  |  |  |  |
 ---  --- 
| 時間帯 | 00-06時 | 06-12時 | 12-18時 | 18-24時 |
| 降水確率  -  20% |

## 明日の天気 - 8月31日(月)

|  |  |  |
 --- 
| 曇時々晴 曇時々晴 | 最高気温 [前日差]  31℃ [ +3 ] | 最低気温 [前日差]  23℃ [ +3 ] |

|  |  |  |  |  |
 ---  --- 
| 時間帯 | 00-06時 | 06-12時 | 12-18時 | 18-24時 |
| 降水確率 | 10% | 0% | 20% | 30% |

## 週間天気 [...] ## 週間天気

|  |  |  |  |  |  |
 ---  ---  --- |
|  | 1日(火) | 2日(水) | 3日(木) | 4日(金) | 5日(土) |
| 天気 | 曇時々晴  曇時々晴 | 曇  曇 | 曇一時雨  曇一時雨 | 曇一時雨  曇一時雨 | 曇  曇 |
| 最高気温 | 33℃ | 35℃ | 31℃ | 27℃ | 27℃ |
| 最低気温 | 24℃ | 25℃ | 23℃ | 22℃ | 20℃ |
| 降水確率 | 30% | 30% | 70% | 60% | 30% |

東京 | 大島 | 八丈島

## 気象庁発表 天気概況
```

この段階ではTavilyによるWeb検索結果を確認しているだけであり、取得した情報はまだQwen3には渡していない。


In [6]:
tavily = TavilyClient()

search_result = tavily.search(
    query="今日の東京の天気",
    max_results=3
)

print(f"title:\n{search_result['results'][0]['title']}\n\n")
print(f"url:\n{search_result['results'][0]['url']}\n\n")
print(f"content:\n{search_result['results'][0]['content']}\n\n")  

title:
東京都 東京の天気


url:
https://mainichi.jp/weather/forecast/city/city_03_01_01.html


content:
# 毎日新聞

 全国の天気
 週間天気予報
 アメダス実況
 降水短時間予報
 衛星雲画像
 天気図
 洗濯情報
 台風
 地震
 警報・注意報
 世界の天気
 花粉情報

# 東京都 東京の天気

9月1日5時発表

## 今日の天気 - 9月1日(火)

|  |  |  |
 --- 
| 曇時々晴 曇時々晴 | 最高気温 [前日差]  30℃ [ +1 ] | 最低気温  - |

|  |  |  |  |  |
 ---  --- 
| 時間帯 | 00-06時 | 06-12時 | 12-18時 | 18-24時 |
| 降水確率  10% | 10% | 20% |

## 明日の天気 - 9月2日(水)

|  |  |  |
 --- 
| 曇時々晴 曇時々晴 | 最高気温  33℃ | 最低気温  24℃ |

|  |  |  |  |  |
 ---  --- 
| 時間帯 | 00-06時 | 06-12時 | 12-18時 | 18-24時 |
| 降水確率 | 10% | 10% | 20% | 20% |

## 週間天気 [...] ## 週間天気

|  |  |  |  |  |  |
 ---  ---  --- |
|  | 3日(木) | 4日(金) | 5日(土) | 6日(日) | 7日(月) |
| 天気 | 曇一時雨  曇一時雨 | 曇時々雨  曇時々雨 | 曇  曇 | 曇  曇 | 曇  曇 |
| 最高気温 | 32℃ | 25℃ | 26℃ | 27℃ | 27℃ |
| 最低気温 | 23℃ | 21℃ | 20℃ | 20℃ | 20℃ |
| 降水確率 | 70% | 70% | 40% | 40% | 40% |

## 東京都その他の地点

東京 | 大島 | 八丈島

## 気象庁発表 天気概況




### 会話履歴を保存するリストの初期化

ユーザーとAIのやり取りを保持するため、会話履歴用のリスト`conversation_history`を用意する。

このリストはチャット処理の開始前に1度だけ初期化し、以降のやり取りを順番に追加していく。


In [7]:
conversation_history = []


### 日付表現を検索できる形へ変換

「今日のニュース」「昨日の天気」のような質問をそのままWeb検索すると、基準日を誤って解釈する場合がある。

そこで、Web検索を行う前に日付表現をPythonで具体的な年月日へ変換する。例えば、2026年9月1日に「昨日のニュース」と入力した場合は、「2026年8月31日のニュース」に変換してから検索する。

`resolve_date_query()`では、今日・昨日・明日などの日付表現に加えて、N日前・N日後、去年・今年・来年などを処理する。すべての自然言語の日付表現への対応を目的とはせず、検索時の代表的な日付誤認を減らすための処理として実装した。


In [8]:
import re

def resolve_date_query(
    user_query: str,
    reference_datetime: datetime
) -> tuple[str, str]:
    """
    ユーザー入力に含まれる日付表現を具体的な日付へ正規化する。

    Returns
    -------
    search_query : str
        Web検索へ渡すための具体化済みクエリ
    date_context : str
        LLMへ渡す日付補足情報
    """

    search_query = user_query
    contexts = []

    # ------------------------------------------------------------
    # すでに西暦が含まれている場合はその情報を優先
    # ------------------------------------------------------------
    explicit_date_pattern = r"\d{4}年\d{1,2}月\d{1,2}日"

    if re.search(explicit_date_pattern, user_query):
        matched = re.search(explicit_date_pattern, user_query).group()
        contexts.append(f"指定日は{matched}です。")
        return search_query, " ".join(contexts)

    # ------------------------------------------------------------
    # 年単位の相対表現
    # ------------------------------------------------------------
    year_replacements = {
        "去年": reference_datetime.year - 1,
        "今年": reference_datetime.year,
        "来年": reference_datetime.year + 1,
    }

    for word, year in year_replacements.items():
        if word in search_query:
            search_query = search_query.replace(
                word,
                f"{year}年"
            )
            contexts.append(
                f"{word}は{year}年です。"
            )

    # N年前 / N年後
    year_before_match = re.search(r"(\d+)年前", search_query)
    if year_before_match:
        years = int(year_before_match.group(1))
        target_year = reference_datetime.year - years

        search_query = search_query.replace(
            year_before_match.group(0),
            f"{target_year}年"
        )

        contexts.append(
            f"{years}年前は{target_year}年です。"
        )

    year_after_match = re.search(r"(\d+)年後", search_query)
    if year_after_match:
        years = int(year_after_match.group(1))
        target_year = reference_datetime.year + years

        search_query = search_query.replace(
            year_after_match.group(0),
            f"{target_year}年"
        )

        contexts.append(
            f"{years}年後は{target_year}年です。"
        )

    # ------------------------------------------------------------
    # 日単位の相対表現
    # ------------------------------------------------------------
    fixed_day_words = {
        "一昨日": -2,
        "昨日": -1,
        "今日": 0,
        "明後日": 2,
        "明日": 1,
    }

    for word, offset in fixed_day_words.items():
        if word in search_query:
            target_date = (
                reference_datetime + timedelta(days=offset)
            ).strftime("%Y年%m月%d日")

            search_query = search_query.replace(
                word,
                target_date
            )

            contexts.append(
                f"{word}は{target_date}です。"
            )

    # N日前
    days_before_match = re.search(r"(\d+)日前", search_query)
    if days_before_match:
        days = int(days_before_match.group(1))

        target_date = (
            reference_datetime - timedelta(days=days)
        ).strftime("%Y年%m月%d日")

        search_query = search_query.replace(
            days_before_match.group(0),
            target_date
        )

        contexts.append(
            f"{days}日前は{target_date}です。"
        )

    # N日後
    days_after_match = re.search(r"(\d+)日後", search_query)
    if days_after_match:
        days = int(days_after_match.group(1))

        target_date = (
            reference_datetime + timedelta(days=days)
        ).strftime("%Y年%m月%d日")

        search_query = search_query.replace(
            days_after_match.group(0),
            target_date
        )

        contexts.append(
            f"{days}日後は{target_date}です。"
        )

    # ------------------------------------------------------------
    # "2025年の10月1日" → "2025年10月1日"
    # ------------------------------------------------------------
    search_query = re.sub(
        r"(\d{4}年)の(\d{1,2}月\d{1,2}日)",
        r"\1\2",
        search_query
    )

    return search_query, " ".join(contexts)


### 質問ごとにWeb検索が必要か判断

すべての質問でWeb検索を実行すると、検索時間が増えるだけでなく、Pythonの基礎知識のように検索が不要な質問まで外部情報へ依存してしまう。

そこで`judge_search_need()`を使い、ユーザーの質問を`Qwen3-1.7B`に判定させる。ニュース・天気・最新情報などはWeb検索を行い、一般知識やプログラミングの基礎質問では検索しない、という振り分けを行う。


In [9]:
def judge_search_need(
    user_query: str,
    current_date: str
) -> bool:

    judge_messages = [
        {
            "role": "system",
            "content": (
                f"現在の日付は{current_date}です。"
                "ユーザーの質問にWeb検索が必要か判定してください。"
                "天気、ニュース、価格、株価、イベント、"
                "最新バージョン、現在の状況など、"
                "現在または最新の情報が必要な場合はYESです。"
                "一方、Pythonの文法、プログラミングの基礎、"
                "用語の意味、数学、一般的な知識など、"
                "学習済みの一般知識だけで回答できる質問はNOです。"
                "Pythonのclass、list、辞書、関数などの"
                "基本的なプログラミング知識についてはNOとしてください。"
                "質問として意味が不明確な入力や、"
                "単なるテスト文字列にもNOと回答してください。"
                "迷った場合はNOとしてください。"
                "回答の最初の行にはYESまたはNOだけを出力してください。"
            )
        },
        {
            "role": "user",
            "content": user_query
        }
    ]

    judge_text = light_tokenizer.apply_chat_template(
        judge_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    judge_inputs = light_tokenizer(
        judge_text,
        return_tensors="pt"
    ).to(light_model.device)

    judge_outputs = light_model.generate(
        **judge_inputs,
        max_new_tokens=16
    )

    judge_generated_ids = judge_outputs[0][
        judge_inputs["input_ids"].shape[1]:
    ]

    judge_response = light_tokenizer.decode(
        judge_generated_ids,
        skip_special_tokens=True
    ).strip().upper()

    judgement = judge_response.splitlines()[0].strip()

    if judgement == "YES":
        return True

    if judgement == "NO":
        return False

    raise ValueError(
        f"想定外の判定結果です: {judge_response}"
    )


### Tavilyから最新情報を取得

Web検索が必要と判定された場合だけ`search_web()`を実行する。

Tavilyから最大3件の検索結果を取得し、タイトル・URL・本文をQwen3へ渡しやすい文字列へ整形する。これにより、ローカルLLM単体では持っていない最新情報を回答の根拠として利用する。


In [10]:
def search_web(search_query: str) -> str:

    search_result = tavily.search(
        query=search_query,
        max_results=3
    )

    search_text = ""

    for result in search_result["results"]:

        search_text += (
            f"タイトル: {result['title']}\n"
            f"URL: {result['url']}\n"
            f"内容: {result['content']}\n\n"
        )

    return search_text


### 質問と検索結果から初回回答を生成

`generate_initial_answer()`では、軽量な`Qwen3-1.7B`を使って最初の回答を生成する。

Web検索を行った場合はTavilyの検索結果を回答材料として渡し、検索しなかった場合はモデル自身の一般知識を利用する。また、これまでの会話履歴も渡すことで、「ここまで何を話したか」といった前の対話を参照する質問にも対応させる。


In [11]:
def generate_initial_answer(
    user_query: str,
    current_date: str,
    date_context: str,
    search_text: str,
    needs_search: bool,
    conversation_history: list
) -> str:

    if needs_search:

        messages = [
            {
                "role": "system",
                "content": (
                    f"現在の日付は{current_date}です。"
                    f"{date_context}"
                    "あなたはWeb検索結果を参考にして回答するAIアシスタントです。"
                    "提供されたWeb検索結果を根拠として、"
                    "ユーザーの質問に日本語で回答してください。"
                    "Web検索結果に記載されていない事実を"
                    "推測して追加しないでください。"
                    "日付表現は与えられた日付情報を基準に判断してください。"
                    "過去の会話履歴は、現在の質問を理解するために"
                    "必要な場合だけ参照してください。"
                    "現在の質問が独立した新しい話題の場合は、"
                    "過去の話題を回答へ持ち込まないでください。"
                    "回答は重要な内容を優先して簡潔にまとめてください。"
                    "ニュースを回答する場合は最大5件程度に絞ってください。"
                )
            }
        ]

        messages.extend(conversation_history)

        messages.append(
            {
                "role": "user",
                "content": (
                    f"ユーザーの質問:\n{user_query}\n\n"
                    f"日付情報:\n{date_context}\n\n"
                    f"Web検索結果:\n{search_text}"
                )
            }
        )

    else:

        messages = [
            {
                "role": "system",
                "content": (
                    f"現在の日付は{current_date}です。"
                    f"{date_context}"
                    "あなたはユーザーの質問に日本語で回答するAIアシスタントです。"
                    "一般的な知識をもとに回答してください。"
                    "プログラミングに関する回答では、"
                    "仕様、メソッド名、構文を正確に記述してください。"
                    "存在しないAPIやメソッドを推測して作らないでください。"
                    "コード例を示す場合は、本文の説明と一致させてください。"
                    "過去の会話履歴は現在の質問に必要な場合だけ参照してください。"
                    "独立した新しい話題なら過去の話題を持ち込まないでください。"
                    "回答は重要な内容を優先して簡潔にまとめてください。"
                )
            }
        ]

        messages.extend(conversation_history)

        messages.append(
            {
                "role": "user",
                "content": user_query
            }
        )

    text = light_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = light_tokenizer(
        text,
        return_tensors="pt"
    ).to(light_model.device)

    outputs = light_model.generate(
        **inputs,
        max_new_tokens=768
    )

    generated_ids = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return light_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


### 別モデルで回答をチェック（Reflection）

小型モデルが生成した回答には、事実誤認や存在しないメソッドの生成などが起こる可能性がある。

そこで`reflect_answer()`では、初回回答を生成した`Qwen3-1.7B`とは別に、より大きい`Qwen3-4B`を評価役として使用する。質問への回答になっているか、検索結果と矛盾していないか、プログラミング上の明らかな誤りがないかなどを確認し、`PASS`または`FAIL`で判定する。

Reflectionによってすべての誤りを検出できるわけではないが、小型モデルだけで回答を確定するよりも品質を確認する工程を追加できる。


In [12]:
def reflect_answer(
    user_query: str,
    current_date: str,
    date_context: str,
    search_text: str,
    initial_response: str
) -> bool:

    reflection_messages = [
        {
            "role": "system",
            "content": (
                "あなたはAIが生成した回答を厳密に評価する役割です。"
                "以下の観点をすべて確認してください。\n"
                "・ユーザーの質問に正しく回答しているか\n"
                "・Web検索結果がある場合、その内容と矛盾していないか\n"
                "・検索結果に存在しない情報を勝手に追加していないか\n"
                "・重要な情報の不足や明らかな誤りがないか\n"
                "・仕様、メソッド名、関数名、引数、戻り値が正しいか\n"
                "・存在しないAPIやメソッドを作っていないか\n"
                "・コード例と本文の説明が矛盾していないか\n"
                "・似た概念を取り違えていないか\n"
                "・現在の質問と無関係な過去の話題が混入していないか\n"
                "・回答が途中で途切れていないか\n"
                "・日付表現が与えられた日付情報と整合しているか\n"
                "事実関係や仕様に少しでも疑わしい点がある場合はFAILです。"
                "すべての観点を満たす場合だけPASSとしてください。"
                "回答の最初の行にはPASSまたはFAILだけを出力してください。"
            )
        },
        {
            "role": "user",
            "content": (
                f"現在の日付:\n{current_date}\n\n"
                f"日付情報:\n{date_context}\n\n"
                f"ユーザーの質問:\n{user_query}\n\n"
                f"Web検索結果:\n{search_text}\n\n"
                f"生成された回答:\n{initial_response}"
            )
        }
    ]

    reflection_text = review_tokenizer.apply_chat_template(
        reflection_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    reflection_inputs = review_tokenizer(
        reflection_text,
        return_tensors="pt"
    ).to(review_model.device)

    reflection_outputs = review_model.generate(
        **reflection_inputs,
        max_new_tokens=32
    )

    reflection_generated_ids = reflection_outputs[0][
        reflection_inputs["input_ids"].shape[1]:
    ]

    reflection_response = review_tokenizer.decode(
        reflection_generated_ids,
        skip_special_tokens=True
    ).strip().upper()

    judgement = reflection_response.splitlines()[0].strip()

    if judgement == "PASS":
        return True

    if judgement == "FAIL":
        return False

    raise ValueError(
        f"想定外のReflection結果です: {reflection_response}"
    )


### 問題が見つかった回答だけ再生成

Reflectionが`FAIL`になった場合は、`regenerate_answer()`で回答を作り直す。

通常の回答は軽量な`Qwen3-1.7B`で処理し、問題が検出された場合だけ`Qwen3-4B`を再生成に使用する。常に大きいモデルで回答するのではなく、通常処理と品質改善処理でモデルの役割を分ける構成とした。


In [13]:
def regenerate_answer(
    user_query: str,
    current_date: str,
    date_context: str,
    search_text: str,
    initial_response: str
) -> str:

    regenerate_messages = [
        {
            "role": "system",
            "content": (
                f"現在の日付は{current_date}です。"
                f"{date_context}"
                "あなたは回答を改善するAIアシスタントです。"
                "元の回答には問題があります。"
                "ユーザーの質問と利用可能なWeb検索結果を再確認してください。"
                "Web検索結果に存在しない事実を推測して追加しないでください。"
                "日付表現は与えられた日付情報を基準にしてください。"
                "プログラミングに関する回答では、"
                "メソッド名、関数名、構文、仕様を正確に確認してください。"
                "存在しないAPIやメソッドを作らないでください。"
                "元の回答の誤りをそのまま引き継がず修正してください。"
                "コード例と本文の説明を一致させてください。"
                "重要な内容だけを簡潔にまとめてください。"
                "ニュースの場合は最大5件程度に絞ってください。"
                "修正後の回答だけを日本語で出力してください。"
            )
        },
        {
            "role": "user",
            "content": (
                f"現在の日付:\n{current_date}\n\n"
                f"日付情報:\n{date_context}\n\n"
                f"ユーザーの質問:\n{user_query}\n\n"
                f"Web検索結果:\n{search_text}\n\n"
                f"修正前の回答:\n{initial_response}"
            )
        }
    ]

    regenerate_text = review_tokenizer.apply_chat_template(
        regenerate_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    regenerate_inputs = review_tokenizer(
        regenerate_text,
        return_tensors="pt"
    ).to(review_model.device)

    regenerate_outputs = review_model.generate(
        **regenerate_inputs,
        max_new_tokens=768
    )

    regenerate_generated_ids = regenerate_outputs[0][
        regenerate_inputs["input_ids"].shape[1]:
    ]

    return review_tokenizer.decode(
        regenerate_generated_ids,
        skip_special_tokens=True
    ).strip()


### 各機能をつないでチャットとして実行

ここまで作成した処理を組み合わせ、ユーザーが繰り返し質問できるチャット形式にする。

`while`ループの中では、入力された質問に対して「日付の変換 → Web検索の判定 → 必要なら検索 → 回答生成 → 回答チェック → 必要なら再生成」という順番で各関数を呼び出す。

検索や回答生成などの長い処理をそれぞれ別の関数に分けたことで、この部分を見るだけでもチャット全体がどの順番で動いているか確認しやすくしている。また、各処理にかかった時間も計測し、Web検索とローカルLLM生成のどこに時間がかかっているか確認できるようにした。


In [ ]:
conversation_history = []


while True:

    user_query = input(
        "質問を入力してください（exitまたは空白で終了）："
    ).strip()

    if user_query == "" or user_query.lower() == "exit":
        print("チャットを終了します。")
        break

    total_start = time.perf_counter()

    reference_datetime = datetime.now()
    current_date = reference_datetime.strftime(
        "%Y年%m月%d日"
    )

    # 日付表現の正規化
    search_query, date_context = resolve_date_query(
        user_query,
        reference_datetime
    )

    # Web検索要否判定
    judge_start = time.perf_counter()

    needs_search = judge_search_need(
        user_query,
        current_date
    )

    judge_time = time.perf_counter() - judge_start

    print(f"Web検索判定: {needs_search}")

    # Web検索
    search_time = 0.0

    if needs_search:

        search_start = time.perf_counter()

        search_text = search_web(
            search_query
        )

        search_time = time.perf_counter() - search_start

        print("Web検索を実行しました。")

        if search_query != user_query:
            print(
                f"検索クエリ: {search_query}"
            )

    else:

        search_text = ""
        print("Web検索は実行しませんでした。")

    # 初回回答
    answer_start = time.perf_counter()

    initial_response = generate_initial_answer(
        user_query=user_query,
        current_date=current_date,
        date_context=date_context,
        search_text=search_text,
        needs_search=needs_search,
        conversation_history=conversation_history
    )

    answer_time = time.perf_counter() - answer_start

    # Reflection
    reflection_start = time.perf_counter()

    reflection_passed = reflect_answer(
        user_query=user_query,
        current_date=current_date,
        date_context=date_context,
        search_text=search_text,
        initial_response=initial_response
    )

    reflection_time = (
        time.perf_counter() - reflection_start
    )

    print(
        f"Reflection: "
        f"{'PASS' if reflection_passed else 'FAIL'}"
    )

    # FAIL時のみ再生成
    regenerate_time = 0.0

    if reflection_passed:

        final_response = initial_response

    else:

        regenerate_start = time.perf_counter()

        final_response = regenerate_answer(
            user_query=user_query,
            current_date=current_date,
            date_context=date_context,
            search_text=search_text,
            initial_response=initial_response
        )

        regenerate_time = (
            time.perf_counter() - regenerate_start
        )

    # 最終回答
    print("\n【回答】")
    print(final_response)
    print()

    # 会話履歴
    conversation_history.append(
        {
            "role": "user",
            "content": user_query
        }
    )

    conversation_history.append(
        {
            "role": "assistant",
            "content": final_response
        }
    )

    # 処理時間
    total_time = (
        time.perf_counter() - total_start
    )

    print("【処理時間】")
    print(
        f"Web検索要否判定: {judge_time:.2f}秒"
    )
    print(
        f"Tavily検索: {search_time:.2f}秒"
    )
    print(
        f"初回回答生成: {answer_time:.2f}秒"
    )
    print(
        f"Reflection: {reflection_time:.2f}秒"
    )

    if not reflection_passed:
        print(
            f"再生成: {regenerate_time:.2f}秒"
        )

    print(
        f"総処理時間: {total_time:.2f}秒"
    )
    print()


質問を入力してください（exitまたは空白で終了）： Pythonの辞書について教えてください


Web検索判定: True
Web検索を実行しました。
Reflection: PASS

【回答】
Pythonの辞書は、キー（ハッシュ）と値のペアでデータを保存するデータ構造です。辞書は、キーを使って値を迅速にアクセスできるため、データ管理に便利です。辞書はリストと異なり、要素はキーで指定され、値はそのキーに対応するデータを表します。辞書の構造は `{キー: 値}` で表され、例として `{'apple': 3, 'pen': 5}` といった形で表されます。辞書には `dict()` と `len()` などの関数やメソッドが用意され、データの追加・削除・検索が可能です。

【処理時間】
Web検索要否判定: 0.34秒
Tavily検索: 1.72秒
初回回答生成: 11.08秒
Reflection: 1.16秒
総処理時間: 14.30秒



質問を入力してください（exitまたは空白で終了）： Pythonのリストとタプルの違いを教えてください


Web検索判定: True
Web検索を実行しました。
Reflection: PASS

【回答】
Pythonのリストとタプルの違いは以下の通りです：

1. **変更可能性**  
   - リストは変更可能（要素を追加・変更・削除可能）  
   - タプルは変更不可能（要素を変更できない）

2. **構造**  
   - リストは `[1, 2, 3]` で表され、リストの要素は順序通りに配置される。  
   - タプルは `(1, 2, 3)` で表され、要素は順序通りに配置され、変更できない。

3. **使いどころ**  
   - リストはデータの追加・変更・削除が必要な場合に使われる。  
   - タプルはデータが変更されない場合や、データの安定性を確保したい場合に使われる。

【処理時間】
Web検索要否判定: 0.39秒
Tavily検索: 1.10秒
初回回答生成: 13.75秒
Reflection: 0.51秒
総処理時間: 15.75秒



質問を入力してください（exitまたは空白で終了）： 今日のニュースを3つ教えてください


Web検索判定: True
Web検索を実行しました。
検索クエリ: 2026年09月01日のニュースを3つ教えてください
Reflection: PASS

【回答】
今日のニュースの3つは以下の通りです：

1. **ウェザーニュース**  
   - 2026年9月1日、西日本は厳しい残暑、北日本は雨が降り始めている。防災DAYに関する最新天気情報が発表されている。

2. **朝日新聞**  
   - 朝刊に「不正融資」に関する記事が掲載され、福井県で浸水被害が発生している。また、サウジに移籍する日本の選手の情報も掲載されている。

3. **インフルエンサーの離婚**  
   - 広島・小園海斗と渡辺リサが離婚を発表。離婚の経緯や理由については発表しないとの声明が掲載されている。

【処理時間】
Web検索要否判定: 0.39秒
Tavily検索: 1.18秒
初回回答生成: 13.31秒
Reflection: 2.07秒
総処理時間: 16.95秒



質問を入力してください（exitまたは空白で終了）： Qwen3の最新情報を教えてください


Web検索判定: True
Web検索を実行しました。
Reflection: FAIL

【回答】
Qwen3の最新情報を以下にまとめます：

1. **Qwen3シリーズのリリース**  
   - 2025年4月からQwen3シリーズがリリースされ、235B-A22B非思考モード、30B-A3B思考モードなど、モデルサイズが拡大されました。  
   - 2025年7月21日にはQwen3-235B-A22B-Instruct-2507がリリースされ、256Kトークンの長文理解をサポートし、モデルカードも公開されました。

2. **Qwen3.8-Maxのリリース**  
   - 2026年7月30日、Qwen3.8-Maxがリリースされ、2.4Tパラメータを達成。  
   - コード生成、研究、ワークなどの長時間タスクに全面的な改善が行われており、オープンウェイグスのリリースも予定されています。

3. **Qwen3シリーズの進化**  
   - Qwen3-235B-A22B、Qwen3-30B-A3Bなどのモデルがリリースされ、モデルサイズと機能面で進化が進んでいます。  
   - 2025年8月にはQwen3-2507がリリースされ、超長文処理が可能となりました。

【処理時間】
Web検索要否判定: 0.41秒
Tavily検索: 2.84秒
初回回答生成: 21.09秒
Reflection: 3.32秒
再生成: 28.65秒
総処理時間: 56.32秒



質問を入力してください（exitまたは空白で終了）： これまでどのような質問をしたでしょうか？


Web検索判定: True
Web検索を実行しました。
Reflection: FAIL

【回答】
### **修正後の回答**

---

### **Pythonの辞書、リスト、タプルの違い**

| パラメータ | リスト | タプル |
|-----------|--------|--------|
| **変更可能** | ✅ はい | ❌ いいえ |
| **構造** | `[1, 2, 3]` | `(1, 2, 3)` |
| **使いどころ** | データの追加・変更・削除 | データの安定性を確保したい場合 |

---

### **今日のニュース（3つ）**

1. **ウェザーニュース**  
   - 2026年9月1日、西日本は厳しい残暑、北日本は雨が降り始めている。

2. **朝日新聞**  
   - 朝刊に「不正融資」に関する記事が掲載され、福井県で浸水被害が発生している。

3. **インフルエンサーの離婚**  
   - 広島・小園海斗と渡辺リサが離婚を発表。離婚の経緯や理由については発表しないとの声明が掲載されている。

---

### **Qwen3の最新情報**

1. **Qwen3シリーズのリリース**  
   - 2025年4月からQwen3シリーズがリリースされ、モデルサイズが拡大されました。

2. **Qwen3.8-Maxのリリース**  
   - 2026年7月30日、Qwen3.8-Maxがリリースされ、2.4Tパラメータを達成。

3. **Qwen3シリーズの進化**  
   - Qwen3-235B-A22B、Qwen3-30B-A3Bなどのモデルがリリースされ、モデルサイズと機能面で進化が進んでいます。

---

### **面接での逆質問の質問例（まとめ）**

- **「何か質問はありますか？」** → 逆質問の質問例を示す
- **「仕事内容を教えてください？」** → データの追加・変更・削除
- **「キャリアモデルについて聞かれる？」** → データの安定性
- **「制度について聞かれる？」** → 資格のアピール
- **「やる気をアピールする？」** → 自己PRのアピール
- **「相性を確認する？」** → 社風との相性
- **「仕事内容を聞かれる？」**

### 処理全体の流れ

このチャットでは、ユーザーの質問をそのままQwen3へ渡すのではなく、質問内容に応じて必要な処理を選択する。

```text
ユーザーが質問
    ↓
日付表現を具体的な日付へ変換
    ↓
Web検索が必要か判定
    ↓
必要な場合だけTavilyで検索
    ↓
Qwen3-1.7Bで初回回答
    ↓
Qwen3-4Bで回答をチェック
    ↓
問題があればQwen3-4Bで再生成
    ↓
回答を表示して会話履歴へ保存
```

この構成により、ローカルLLMだけでは不足する最新情報をWeb検索で補いながら、不要な検索を減らし、生成後にも回答を確認する仕組みを試した。

一方で、小型LLMによる検索要否判定やReflectionには揺らぎがあり、自然言語の日付表現にも対応範囲がある。これらは実行結果を確認したうえで、本Notebookの最後に課題として整理する。


## 動作検証と総括

最後に、Web検索が不要と考えられる質問、最新情報を必要とする質問、会話履歴を参照する質問を同一セッションで実行し、各機能の動作を確認した。

### 動作検証で確認できたこと

- 「今日のニュースを3つ教えてください」では、「今日」を実行日の `2026年09月01日` に変換してからTavily検索を実行できた。相対的な日付表現を具体的な日付へ変換する処理が機能していることを確認できた。
- 「Qwen3の最新情報を教えてください」ではWeb検索が実行され、初回回答に対してReflectionが `FAIL` を返したため、Qwen3-4Bによる再生成まで処理が進んだ。検索・回答生成・Reflection・再生成という一連の処理が機能していることを確認できた。
- 最後に「これまでどのような質問をしたでしょうか？」と質問したところ、それまでに行ったPython、ニュース、Qwen3に関する内容を回答に含めた。このことから、`conversation_history` に保存した過去の対話を後続の回答で参照できることを確認できた。
- 各処理時間を計測したことで、Tavily検索そのものよりもローカルLLMによる回答生成や再生成に時間がかかることを確認できた。特にReflectionで `FAIL` となった履歴確認の質問では、再生成を含めた総処理時間が146.82秒となった。

### 動作検証で確認できた課題

今回の検証では、意図した通りに動作しないケースも確認できた。

まず、「Pythonの辞書について教えてください」「Pythonのリストとタプルの違いを教えてください」のような一般的な知識で回答できる質問でも、Web検索要否判定が `True` となった。検索要否を小型LLMに判断させているため、プロンプトで条件を指定しても判定には揺らぎがある。

また、会話履歴を確認する質問でもWeb検索が実行された。さらに、過去のPython・ニュース・Qwen3に関する質問は参照できた一方で、実際には質問していない「面接での逆質問」に関する内容も生成された。Reflectionはこの回答を `FAIL` と判定したものの、再生成後にも不要な内容が残った。これにより、Reflectionを追加するだけでは誤生成を完全に防げないことも確認できた。

Web検索結果を利用する回答についても、検索結果の内容や取得順位によって最終回答の品質が左右される。そのため、Web検索を行うこと自体が回答の正確性を保証するものではなく、検索結果の選別や情報源の評価も重要になる。

### 今後の改善点

今後さらに改善する場合は、Web検索要否判定をLLMだけに任せずルールベースの条件と組み合わせる方法、会話履歴から現在の質問に必要な情報だけを抽出する方法、Reflectionで `PASS / FAIL` だけでなく問題点や修正理由も生成して再生成へ渡す方法などが考えられる。また、Web検索結果の情報源や関連度を評価してからLLMへ渡すことで、検索結果に起因する回答品質のばらつきを抑えられる可能性がある。

### まとめ

今回の実装では、ローカル環境で動作するQwen3を中心に、TavilyによるWeb検索、検索要否判定、相対日付の変換、会話履歴、Reflection、再生成を組み合わせたチャットを構築した。軽量なQwen3-1.7Bを検索判定と初回回答に使用し、Qwen3-4BをReflectionと再生成に使用することで、処理内容に応じてモデルを使い分ける構成とした。

動作検証を通して、一連の処理が実際に連携して動くことを確認できた一方、小型LLMによる検索要否判定の揺らぎ、Web検索結果への依存、Reflection後にも誤生成が残る可能性、ローカルLLMで複数回生成する場合の処理時間増加といった課題も確認できた。

単にWeb検索を追加するだけではなく、「いつ検索するか」「生成した回答をどのように確認するか」「品質向上と処理時間をどう両立するか」を考えながら実装したことで、ローカルLLMを利用したチャットシステムにおける制御方法と、その限界を実際の動作結果から確認できた。

